# Assignment 2 — Question 1: Used-car data preparation and manipulation

**Dataset:** `data_raw/train.csv` (provided with assignment)  
**Target:** `Price` in lakhs  
**Tasks:** (a) missing values; (b) unit removal; (c) one-hot encoding; (d) engineered feature; (e) select/filter/rename/mutate/arrange/grouped summaries.

Run cells top-to-bottom. All file locations are resolved relative to either repository root or `src/`. The original source file is never overwritten.

In [3]:
import sys
import ipykernel

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Kernel:", ipykernel.__version__)

Python: 3.12.13 (main, Apr  5 2026, 20:59:31) [GCC 11.3.0]
Executable: /home/cstever/college/comp_sci_5530/Assignment_2_Project/.venv/bin/python
Kernel: 7.4.0


In [4]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'data_raw' / 'train.csv').exists() else Path.cwd().parent
assert (ROOT / 'data_raw' / 'train.csv').exists(), 'Place train.csv under data_raw/ and run from repository root or src/.'
df = pd.read_csv(ROOT / 'data_raw' / 'train.csv')
print('Input shape:', df.shape)
display(df.head())

Input shape: (5847, 14)


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


## (a) Missing values and justified treatment

Inspect **all** columns before making changes. We drop the unnamed index-like field, which is not a car attribute. Missing `Mileage`, `Engine`, `Power`, and `Seats` are filled with the **median** of observed values because these measurements can be skewed and the missing fractions are small. All unit conversions occur **before** imputation. `New_Price` is missing for most listings, so median imputation would create misleading artificial original prices. To meet the assignment’s requirement to handle missing values without deleting most rows, replace missing `New_Price` with the observed median **and** add a binary `New_Price_Missing` indicator. Because more than 80% of the column is missing, these imputed values are placeholders rather than reliable actual original prices; avoid using that field for substantive conclusions. The target `Price` has no missing values; if it did, we would drop those rows rather than invent target labels.

In [5]:
missing_before = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_percent': 100*df.isna().mean()}).sort_values('missing_count', ascending=False)
display(missing_before)
print('Duplicate full records:', df.duplicated().sum())

,missing_count,missing_percent
New_Price,5032,86.061228
Seats,38,0.649906
Power,36,0.615700
Engine,36,0.615700
Mileage,2,0.034206
Unnamed: 0,0,0.000000
Fuel_Type,0,0.000000
Kilometers_Driven,0,0.000000
Year,0,0.000000
Location,0,0.000000


Duplicate full records: 0


## (b) Clean numeric attributes and normalize units

Mileage comes in `kmpl` and `km/kg`. Both units are removed to create numeric values, **but these units measure different quantities**; a `Mileage_Unit` column is retained so they are not implicitly treated as directly comparable. `Engine` is in cubic centimeters, and `Power` in brake horsepower. For `New_Price`, prices in `Cr` are converted to lakhs (1 crore = 100 lakhs); `Lakh` remains unchanged. Source numerical measurements are retained in appropriate units.

In [6]:
clean = df.drop(columns=['Unnamed: 0'], errors='ignore').copy()
clean['Mileage_Unit'] = clean['Mileage'].astype('string').str.extract(r'(kmpl|km/kg)', expand=False).fillna('Unknown')
for field in ['Mileage', 'Engine', 'Power']:
    clean[field] = pd.to_numeric(clean[field].astype('string').str.extract(r'([0-9]+(?:\.[0-9]+)?)',expand=False), errors='coerce')

original_new_price = clean['New_Price'].astype('string')
new_value = pd.to_numeric(original_new_price.str.extract(r'([0-9]+(?:\.[0-9]+)?)',expand=False),errors='coerce')
new_unit = original_new_price.str.extract(r'(?i)\b(Lakh|Cr)\b',expand=False).str.lower()
clean['New_Price'] = np.where(new_unit.eq('cr').fillna(False),new_value*100,new_value)
clean['New_Price_Missing'] = clean['New_Price'].isna().astype('int8')
new_price_missing_before=int(clean['New_Price_Missing'].sum())
clean['New_Price'] = clean['New_Price'].fillna(clean['New_Price'].median())
assert clean['New_Price'].dropna().ge(0).all()
for field in ['Mileage','Engine','Power','Seats']:
    clean[field] = clean[field].fillna(clean[field].median())
# Unit-aware mileage median: only three km/kg listings; preserve original unit marker for interpretation.
print('Missing values after treatment:')
display(clean.isna().sum().to_frame('missing_count'))
print('Numeric fields after cleanup:')
display(clean[['Mileage','Engine','Power','Seats','New_Price','Mileage_Unit']].head(8))
print('New_Price missing values filled with median:', new_price_missing_before)
assert clean.isna().sum().sum() == 0, "Inspect remaining missing values"

Missing values after treatment:


,missing_count
Name,0
Location,0
Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,0
Engine,0
Power,0


Numeric fields after cleanup:


,Mileage,Engine,Power,Seats,New_Price,Mileage_Unit
0,19.67,1582,126.2,5.0,11.75,kmpl
1,13.0,1199,88.7,5.0,8.61,km/kg
2,20.77,1248,88.76,7.0,11.75,kmpl
3,15.2,1968,140.8,5.0,11.75,kmpl
4,23.08,1461,63.1,5.0,11.75,kmpl
5,11.36,2755,171.5,8.0,21.00,kmpl
6,20.54,1598,103.6,5.0,11.75,kmpl
7,22.3,1248,74.0,5.0,11.75,kmpl


New_Price missing values filled with median: 5032


## (c) One-hot encode Fuel_Type and Transmission

Use separate 0/1 indicators for every observed category (`drop_first=False`), preserving all information. If the encoded data are later used in a linear regression with an intercept, one indicator per group would normally be removed to prevent perfect multicollinearity.

In [7]:
categorical_columns=['Fuel_Type','Transmission']
encoded = pd.get_dummies(clean,columns=categorical_columns,prefix=categorical_columns,dtype=int,drop_first=False)
encoded_columns=[c for c in encoded if c.startswith('Fuel_Type_') or c.startswith('Transmission_')]
print('One-hot indicators:', encoded_columns)
display(encoded[encoded_columns].head())
assert (encoded[[c for c in encoded_columns if c.startswith('Fuel_Type_')]].sum(axis=1)==1).all()
assert (encoded[[c for c in encoded_columns if c.startswith('Transmission_')]].sum(axis=1)==1).all()

One-hot indicators: ['Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual']


,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
0,1,0,0,0,1
1,0,0,1,0,1
2,1,0,0,0,1
3,1,0,0,1,0
4,1,0,0,0,1


## (d) Engineer car age

Age is computed using the **execution year**, not a hard-coded calendar year, so the notebook remains reproducible in future years (the analysis date is reported). Source manufacturing year is preserved.

In [8]:
analysis_year=datetime.now().year
encoded['Car_Age'] = analysis_year - encoded['Year']
print('Analysis year:',analysis_year)
display(encoded[['Name','Year','Car_Age']].head())
assert (encoded.Car_Age>=0).all(), "Inspect future-dated manufacturing years."

Analysis year: 2026


,Name,Year,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,2015,11
1,Honda Jazz V,2011,15
2,Maruti Ertiga VDI,2012,14
3,Audi A4 New 2.0 TDI Multitronic,2013,13
4,Nissan Micra Diesel XV,2013,13


## (e) Python equivalents: select, filter, rename, mutate, arrange, and group-by summarize

The examples are separate transformations of `encoded`, so each operation can be evaluated independently.

In [9]:
# select: subset the most relevant attributes
selected = encoded[['Name','Location','Year','Car_Age','Price']]
print('SELECT'); display(selected.head())

# filter: focus on cars sold for at least 10 lakh with no more than two previous decades of age
filtered = encoded.loc[(encoded['Price']>=10)&(encoded['Car_Age']<=20)].copy()
print('FILTER rows:',len(filtered)); display(filtered[['Name','Price','Car_Age']].head())

# rename: choose a descriptive target label
renamed = encoded.rename(columns={'Price':'Used_Price_Lakh','Kilometers_Driven':'Odometer_Km'})
print('RENAME'); display(renamed[['Used_Price_Lakh','Odometer_Km']].head())

# mutate: price per 1,000 kilometers driven (avoid dividing by zero)
mutated = encoded.assign(Price_Per_1000_Km=lambda x: np.where(x['Kilometers_Driven']>0,1000*x['Price']/x['Kilometers_Driven'],np.nan))
print('MUTATE');display(mutated[['Price','Kilometers_Driven','Price_Per_1000_Km']].head())

# arrange: highest to lowest price, then newer vehicles
arranged = encoded.sort_values(['Price','Year'],ascending=[False,False])
print('ARRANGE');display(arranged[['Name','Year','Price']].head())

# group by and summarize, grouping by transmission indicator
transmission_name = np.select([encoded[c].eq(1) for c in encoded_columns if c.startswith('Transmission_')], [c.replace('Transmission_','') for c in encoded_columns if c.startswith('Transmission_')],default='Other')
summary_input = encoded.assign(Transmission_Category=transmission_name)
grouped = (summary_input.groupby('Transmission_Category',dropna=False)
           .agg(Listings=('Price','size'), Mean_Price_Lakh=('Price','mean'), Median_Price_Lakh=('Price','median'), Mean_Age=('Car_Age','mean'))
           .round(2).sort_values('Mean_Price_Lakh',ascending=False))
print('GROUP BY + SUMMARIZE');display(grouped)
assert grouped.Listings.sum()==len(encoded)

SELECT


,Name,Location,Year,Car_Age,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,11,12.50
1,Honda Jazz V,Chennai,2011,15,4.50
2,Maruti Ertiga VDI,Chennai,2012,14,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,13,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,13,3.50


FILTER rows: 1493


,Name,Price,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,12.50,11
3,Audi A4 New 2.0 TDI Multitronic,17.74,13
5,Toyota Innova Crysta 2.8 GX AT 8S,17.50,10
11,Land Rover Range Rover 2.2L Pure,27.00,12
12,Land Rover Freelander 2 TD4 SE,17.50,14


RENAME


,Used_Price_Lakh,Odometer_Km
0,12.50,41000
1,4.50,46000
2,6.00,87000
3,17.74,40670
4,3.50,86999


... (2 outputs left out)

ARRANGE


,Name,Year,Price
3952,Land Rover Range Rover 3.0 Diesel LWB Vogue,2017,160.00
5620,Lamborghini Gallardo Coupe,2011,120.00
5752,Jaguar F Type 5.0 V8 S,2015,100.00
1457,Land Rover Range Rover Sport SE,2019,97.07
1917,BMW 7 Series 740Li,2018,93.67


GROUP BY + SUMMARIZE


,Listings,Mean_Price_Lakh,Median_Price_Lakh,Mean_Age
Transmission_Category,,,,
Automatic,1712,19.89,16.00,12.12
Manual,4135,5.42,4.65,12.73


## Export and interpretation

The final table contains encoded variables, age, numeric-valued measurements, and the original price target. Missing original `New_Price` values are median-imputed and explicitly flagged; preserve `Mileage_Unit`. All reported summary differences are descriptive, not causal.

In [10]:
out=ROOT/'data_clean'/'used_cars_cleaned.csv'
encoded.to_csv(out,index=False)
print('Saved:',out.relative_to(ROOT),'| rows:',len(encoded),'| columns:',len(encoded.columns))
print('Original New_Price missing percentage:',f"{100*encoded['New_Price_Missing'].mean():.1f}%")
print('Original target mean (lakh):',round(df.Price.mean(),3),'| cleaned target mean:',round(encoded.Price.mean(),3))
assert len(encoded)==len(df) and encoded.Price.equals(df.Price)
assert encoded[['Mileage','Engine','Power','Seats']].notna().all().all()

Saved: data_clean/used_cars_cleaned.csv | rows: 5847 | columns: 19
Original New_Price missing percentage: 86.1%
Original target mean (lakh): 9.654 | cleaned target mean: 9.654
